# Chapter 16 — Debugging Evaluation

**Book alignment:** Debugging AI From First Principles, Chapter 16

**Question this notebook isolates:** Validation accuracy 0.97, intent-slice 0.55, weights
frozen. Do three perturbations of the *instrument* — seed swap, fresh slice, metric swap —
with pre-written score movements separate leakage/split-rot (**H1**), metric-vs-intent
mismatch (**H2**), and seed luck (**H3**)?

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# a 95/5 imbalanced validation set; the model has modest real skill on the minority class
n_neg, n_pos = 1900, 100
y = np.r_[np.zeros(n_neg, int), np.ones(n_pos, int)]
pred = y.copy()
pred[rng.choice(np.where(y == 0)[0], 20, replace=False)] = 1     # 20 false positives
pred[rng.choice(np.where(y == 1)[0], 55, replace=False)] = 0     # 55 false negatives (minority mostly missed)

def accuracy(a, b):          return float((np.asarray(a) == np.asarray(b)).mean())
def balanced_accuracy(t, p):
    t, p = np.asarray(t), np.asarray(p)
    return float(np.mean([ (p[t == c] == c).mean() for c in np.unique(t) ]))

## 1. The headline vs. the baseline it must beat

In [ ]:
headline = accuracy(y, pred)
majority = accuracy(y, np.zeros_like(y))
print(f"headline accuracy      : {headline:.3f}")
print(f"majority-class baseline : {majority:.3f}")
assert abs(headline - majority) < 0.03
print("the score barely beats 'always predict 0' - aggregation is anesthesia")

## 2. Perturb the instrument (weights frozen), predictions pre-written

In [ ]:
# H3 seed swap: 3 re-splits of the same pool -> prediction if luck: score swings a lot
seeds = []
for s in range(3):
    r = np.random.default_rng(s)
    keep = r.choice(len(y), 1500, replace=False)
    seeds.append(accuracy(y[keep], pred[keep]))
print("seed swap x3:", [round(v, 3) for v in seeds], " spread", round(max(seeds) - min(seeds), 3))
assert max(seeds) - min(seeds) < 0.03            # stable -> H3 exonerated

# H1 fresh slice: an independently sourced, balanced, overlap-free slice
fresh_t = np.r_[np.zeros(300, int), np.ones(300, int)]
fresh_p = fresh_t.copy()
fresh_p[rng.choice(np.where(fresh_t == 1)[0], 175, replace=False)] = 0   # ~0.58 balanced
fresh_p[rng.choice(np.where(fresh_t == 0)[0], 75, replace=False)] = 1
fresh = accuracy(fresh_t, fresh_p)
print(f"fresh slice accuracy: {fresh:.3f}")
assert headline - fresh > 0.25                   # large collapse -> H1 live

# H2 metric swap: same predictions, intent-aligned metric
bal = balanced_accuracy(y, pred)
print(f"balanced accuracy (identical predictions): {bal:.3f}")
assert headline - bal > 0.15                     # drop with unchanged predictions -> H2 live

## 3. Repair the instrument, then (and only then) return to the model

In [ ]:
REPAIRED_EVAL = {
    "metric": "balanced_accuracy",
    "slice_floors": {"minority_class": 0.70, "fresh_slice": 0.70},
    "split_rule": "regenerate per release; assert zero train/val ID overlap",
    "report": "3 seeds, spread published",
}
for k, v in REPAIRED_EVAL.items():
    print(f"{k}: {v}")
print("\nH1 + H2 jointly convicted; H3 exonerated. zero retraining until the exam is fixed.")

## What we earned

An evaluation is a measurement device with its own error model, and it debugs like one:
perturb the device, predict the reading's movement, observe. Weights never moved. The seed
swap stayed within 0.03 (luck exonerated); the fresh, balanced slice collapsed 0.97 → ~0.58
(leakage/rot); the metric swap dropped the headline to ~0.71 on *identical predictions*
(metric-vs-intent mismatch). The repair is a pinned metric with slice floors and a
regenerated split — not a bigger model.

**Notebook 17 / Chapter 17** crosses into Part IV: the model is behind glass, and the
diagnosis has to move to the boundary.